# Phase 1: Data Exploration and Preprocessing

**Project:** A comparative study of RAG techniques for generating image captions
**Dataset:** MS COCO (train2017 and val2017)

## 1. Project Idea

### 1.1 What is the problem?

Image captioning means making a computer write a short sentence that describes the content of an image.

Modern AI models can generate good captions, but they can still make mistakes:

- They miss objects that are present in the image.
- They invent objects that are not present in the image (hallucination).
- They are not always precise about colors, details, and relationships between objects.

### 1.2 How can RAG solve this problem?

RAG (Retrieval-Augmented Generation) helps the model by providing additional information before it generates a caption.

It works in three steps:

1. **Retrieve:** Search a database for images and captions similar to the input image.
2. **Augment:** Add the retrieved information to the model's prompt.
3. **Generate:** The model generates a caption using the image and the additional context.

### 1.3 What is the goal of this project?

We want to compare different RAG techniques using the same dataset.

We evaluate the techniques according to:

- Caption quality.
- Hallucination rate.
- Execution time.

## 2. The Dataset: MS COCO

### 2.1 What is COCO?

MS COCO (Common Objects in Context) is a large public dataset containing everyday images with objects appearing in natural scenes.

It is widely used for object detection, segmentation, and image captioning.

COCO provides three types of information:

- **Objects:** Each annotated object has a category, a bounding box, and a segmentation mask.
- **Captions:** Human-written sentences describing the images.
- **Categories:** 80 object classes grouped into 12 super-categories.

### 2.2 Dataset structure

COCO contains image directories and annotation files.

| File | Content |
|---|---|
| `train2017/` | Training images |
| `val2017/` | Validation images used as the source of the final test subset |
| `instances_*.json` | Object categories, bounding boxes, and segmentation annotations |
| `captions_*.json` | Human-written image captions |

The `image_id` field links an image to its corresponding annotations and captions.

### 2.3 Splits used in this project

| Subset | Size | Source | Purpose |
|---|---|---|---|
| Knowledge base | 10,000 images | train2017 | Database used for retrieval |
| Validation | 100 images | train2017 | Development and parameter tuning |
| Test | 1000 images | val2017 | Final evaluation |

> **Anti-leakage rule:** Test images and their reference captions must never be included in the knowledge base used for retrieval. Validation musnt collide with 10000 of training 

## 3. Phase 1: Data Exploration and Preprocessing

### 3.1 Objective

Before building the RAG systems, we need a clean, consistent, and reproducible dataset.

Phase 1 prepares the raw COCO data for the later retrieval and caption-generation stages.

It includes four main steps:

| Step | Focus | Main outputs |
|---|---|---|
| Step 1 | Ingestion and environment setup | Raw data, repository, integrity reports |
| Step 2 | Splits, anti-leakage, and metadata | Split JSON files, metadata, caption lookup files |
| Step 3 | Transformation and statistics | Processed images, processed captions, reports |
| Step 4 | Vectorization, indexing, and QA | Image and text embeddings, FAISS index, validation reports |

The project is divided among four team members, with each step containing several tasks.

# Step 1: Ingestion and Environment Setup

## Task 1.1: Download the COCO data

**Goal:** Obtain the 10,100 training images and their captions, ensuring that all team members use the same subset.

**Script:** `scripts/download_coco_rag_subset.py`

Before running the script, download and extract `annotations_trainval2017.zip` from the COCO website.

The script needs the file `annotations/captions_train2017.json`.

**What the script does:**

- Reads `captions_train2017.json`, which contains caption annotations for approximately 118,000 training images.
- Sorts the images by ID.
- Selects 10,000 random images using a fixed seed (`20261027`).
- Because the selection uses sorting and a fixed seed, every team member obtains the same subset.
- Collects the captions associated with the selected images.
- Saves the selection in `train_subset_10000.json`, ensuring that rerunning the process does not change the selected dataset.
- Downloads the selected images into `train2017_subset/`.
- The script retries failed downloads up to four times and skips files that already exist, allowing the process to be resumed.

**Output files:**

| File | Content |
|---|---|
| `train_subset_10100.json` | Frozen selection containing image IDs, filenames, URLs, and captions |
| `train_subset_10100_filenames.txt` | One selected filename per line |
| `captions_metadata_10100.json` | Metadata associated with the selected images |
| `train2017_subset/` | Downloaded training images |
| `failed_downloads.json` | Created only if some downloads fail |

**Command:**

```bash
python scripts/download_coco_rag_subset.py --annotations-dir . --output-dir coco_rag_data
```

## Task 1.2: Repository setup

**Goal:** Provide the team with a shared development environment.

**What we did:**

- Created a shared GitHub repository so all four members can collaborate.
- Added a `.gitignore` file to prevent large generated files, such as images and embeddings, from being pushed unnecessarily.
- Added a `requirements.txt` file containing the Python dependencies required by the project.

## Task 1.3: File integrity audit

**Goal:** Ensure that every selected image exists and can be opened before using it. (Knowledge base)

**Script:** `scripts/diagnose_images.py`

**What the script does:**

- Reads the expected filenames from `train_subset_10100.json`.
- Compares the expected files with the actual files in `train2017_subset/`.
- Identifies missing images and unexpected images.
- Checks every image using two operations:
  - `image.verify()` checks whether the image file structure is valid.
  - `image.load()` decodes the image pixels and detects additional corruption problems.
- Saves the results in `diagnostic_report.json`.
- The script reports problems without deleting or modifying the original files.

**Output:** `diagnostic_report.json`

The final status is either `PASS` or `ISSUES_FOUND`.

**Command:**

```bash
python scripts/diagnose_images.py \
    --images-dir coco_rag_data/train2017_subset \
    --selection-file coco_rag_data/train_subset_10100.json \
    --report coco_rag_data/diagnostic_report.json
```

## Task 1.4: Verify dataset counts

**Goal:** Verify that the selected dataset is complete and consistent.

**What we check:**

- The selection contains exactly 10,100 images.
- There are no duplicate image IDs.
- Every selected image exists on disk.
- Every selected image has at least five associated captions.

The validation and test subsets are checked during Step 2.

# Step 2: Splits, Anti-Leakage and Metadata

## 2.1 Objective

The objective of this step is to organize the selected COCO data into separate subsets and prepare the metadata and caption resources needed by the retrieval system.

The main priorities are:

- Creating reproducible dataset partitions.
- Separating knowledge-base data from evaluation data.
- Preparing object and caption information.
- Building structured lookup resources.
- Checking the consistency of the prepared data.

## Task 2.1: Dataset partitioning and split creation

**Goal:** Create the knowledge-base, validation, and test partitions.

**Script:** `scripts/create_splits.py`

**What the script does:**

- Loads the selected training dataset from `train_subset_10100.json`, `val2017`
- Creates the knowledge-base and validation split files.
- Prepares separate records for the final test subset.
- Saves the partitions as JSON files for later processing.

**Main output files:**

| File | Content |
|---|---|
| `train_kb_ids.json` | 10000 Image IDs and records belonging to the knowledge base |
| `val_ids.json` | 100 Image IDs selected for validation |
| `test_ids.json` | 1000 Image IDs selected for final testing |

These files provide a consistent reference for the later preprocessing and embedding stages.

**Command:**

```bash
python scripts/create_splits.py
```

> 

## Task 2.2: Test metadata preparation

**Goal:** Prepare structured information about the objects appearing in the test images.

**Script:** `scripts/build_unified_lookup.py`

**What the script does:**

- The script builds metadata associated with the selected test images using COCO annotation information.
- The resulting metadata provides a structured representation of the objects associated with each test image.

**Output file:** `test_lookup.json`

This file can be used during later evaluation and analysis to inspect the objects associated with test images.

## Task 2.3: Unified metadata and caption lookup

**Goal:** Organize image information, captions, and object annotations into a unified structure.

**Script:** `scripts/build_unified_lookup.py`

**What the script does:**

- The script builds a unified lookup resource connecting selected images with their available information.
- This makes it easier for later pipeline components to access image-related information through image IDs rather than repeatedly searching through separate annotation files.
- The lookup resource is also useful for checking the consistency of the selected dataset and its associated annotations.

**Output file:** `unified_lookup.json`

Additional caption and metadata resources are organized in the following files:

| File | Purpose |
|---|---|
| `captions_metadata_10000.json` | Caption metadata associated with the selected training subset |
| `captions_kb.json` | Knowledge-base caption records |
| `captions_kb_flat.jsonl` | Flattened knowledge-base caption corpus |
| `captions_test.json` | Test caption records |
| `captions_test_flat.jsonl` | Flattened test caption corpus |

The separation between knowledge-base captions and test captions supports the project's anti-leakage organization.

## Task 2.4: Integration testing

**Goal:** Check the consistency of the prepared Phase 1 data before moving to image and caption transformation.

**Script:** `scripts/test_phase1_integration.py`

**What the script does:**

- The integration test checks the compatibility of the prepared project resources.
- It is used to verify that the different components prepared during the initial stages can work together before the next processing stages.

**Main purpose:**

- Check the consistency of the prepared dataset resources.
- Identify missing or inconsistent data.
- Verify that the required files are available for the next pipeline stages.

**Command:**

```bash
python scripts/test_phase1_integration.py
```


# Step 3: Transformation and Statistics

## 3.1 Objective

After preparing the dataset partitions and metadata, the next step is to standardize the images and captions.

The objective is to make the data consistent and suitable for the embedding models and retrieval pipeline.

This step includes image preprocessing, caption processing, batch execution, and validation.

## Task 3.1: Image preprocessing and normalization

**Goal:** Standardize the visual inputs before passing them to the image encoder.

**Main scripts:**

- `scripts/image_pipeline.py`
- `scripts/preprocess_images.py`

**What the preprocessing pipeline does:**

The image preprocessing stage prepares the images for the downstream vision models.

The intended operations include:

- Correcting image orientation using EXIF information.
- Converting images into RGB format.
- Resizing and cropping images to the required dimensions.
- Saving the standardized image files.

The standardized image dimensions are **224 × 224 pixels**, suitable for the selected CLIP image encoder.

**Main directories:**

| Directory | Purpose |
|---|---|
| `train2017_subset/` | Original selected training images |
| `processed_images/` | Processed image outputs |
| `processed_images_batch/` | Images processed through batch execution |
| `processed_images_test/` | Processed test images |

**Report:** `image_transformation_report.json`

This report records information about the image transformation process.

## Task 3.2: Caption preprocessing

**Goal:** Prepare caption data for later retrieval and analysis.

**Script:** `scripts/caption_pipeline.py`

The caption processing pipeline prepares the caption resources used by the project.

The project maintains separate caption resources for the knowledge base and test data.

**Main outputs:**

| File | Purpose |
|---|---|
| `captions_kb.json` | Knowledge-base captions |
| `captions_kb_flat.jsonl` | Flattened knowledge-base caption corpus |
| `captions_test.json` | Test reference captions |
| `captions_test_flat.jsonl` | Flattened test caption corpus |
| `caption_cleaning_report.json` | Caption processing report |
| `caption_changes.json` | Record of caption changes |
| `captions_kb_report.json` | Knowledge-base caption report |
| `captions_test_report.json` | Test caption report |

The separation between knowledge-base and test captions is important because test reference captions must remain reserved for final evaluation rather than being used as retrieval information.

The original evaluation references should be preserved independently from any normalized text used for retrieval.

## Task 3.3: Batch image preprocessing

**Goal:** Process the image collection efficiently and consistently.

**Script:** `scripts/run_preprocessing_batch.py`

Processing a large image dataset one file at a time can be time-consuming. The batch-processing script organizes the preprocessing work into manageable groups.

**What the script does:**

- Processes images in batches.
- Tracks processing progress.
- Records successful and unsuccessful operations.
- Generates a summary of the execution.

**Output:** `batch_processing_report.json`

This report provides information about the batch execution, including the number of processed, skipped, and failed images.

## Task 3.4: Processed-image validation

**Goal:** Verify that the processed images meet the required format and dimensions.

**Script:** `scripts/validate_processed_images.py`

The validation stage checks the generated image files after preprocessing.

It is used to detect problems such as:

- Missing or unreadable images.
- Incorrect image formats.
- Incorrect dimensions.
- Images that do not use the expected RGB format.

**Main outputs and resources:**

- `processed_images/`
- `processed_images_batch/`
- `processed_images_test/`
- `image_transformation_report.json`
- `batch_processing_report.json`

These resources provide the basis for checking that the image dataset is ready for vectorization.

# Step 4: Vectorization, Indexing and Quality Assurance

## 4.1 Objective

The objective of this step is to convert the prepared images and captions into numerical representations called **embeddings**.

These embeddings allow the RAG system to compare images and text using vector similarity.

The final objective is to prepare the vector database used during retrieval.

## Task 4.1: Image embedding generation

**Goal:** Convert processed images into dense numerical vectors.

**Script:** `scripts/extract_image_embeddings.py`

The image embedding stage uses a pretrained vision encoder to extract feature vectors from the processed images.

The selected image encoder is: `openai/clip-vit-base-patch32`

The extraction process operates in batches to reduce memory consumption.

The generated image embeddings represent visual information in a numerical format suitable for similarity comparison.

**Main directories:**

| Directory | Purpose |
|---|---|
| `image_embeddings/` | Image embedding outputs |
| `image_embeddings_test/` | Test image embedding outputs |

The embedding files and their mappings connect the generated vectors to their corresponding images.

## Task 4.2: Text embedding generation

**Goal:** Convert caption text into numerical vectors.

**Script:** `scripts/extract_text_embeddings.py`

The text embedding stage converts the prepared caption corpus into vector representations.

These embeddings allow the system to compare textual information and use it during retrieval.

The generated vectors are associated with their corresponding image or caption records through mapping information.

**Main outputs:**

| File | Purpose |
|---|---|
| `text_embeddings_mapping.json` | Mapping between text embeddings and their source records |
| `text_embedding_report.json` | Summary of text embedding generation |

The text embedding outputs are intended to support text-based retrieval and later comparisons between retrieval strategies.

## Task 4.3: FAISS vector database construction

**Goal:** Build an efficient vector index for similarity search.

**Script:** `scripts/build_faiss_index.py`

FAISS is used to organize vector representations into an index that can be queried efficiently.

The index allows the system to retrieve the most similar vectors for a given query.

The general process consists of:

1. Loading the generated embedding vectors.
2. Initializing the selected FAISS index.
3. Adding the vectors to the index.
4. Maintaining the association between vector positions and their original image or caption records.
5. Saving the index for later retrieval.

The selected FAISS indexing method and its configuration should be specified according to the actual implementation.

**Expected output:** A serialized FAISS index containing the knowledge-base vectors.

The index and its mapping information will be used by the later RAG retrieval components.

## Task 4.4: Final quality assurance

**Goal:** Verify that the generated Phase 1 resources are consistent and ready for the next development phase.

**Main validation script:** `scripts/test_phase1_integration.py`

The final validation should cover the availability and consistency of the generated resources, including:

- Dataset partitions.
- Image preprocessing outputs.
- Caption resources.
- Image embeddings.
- Text embeddings.
- Mapping information.
- Vector index, once generated.

The purpose is to identify missing files, incorrect dimensions, inconsistent mappings, or other problems before the RAG implementation begins.



## 4. Main Project Directory Organization

The Phase 1 implementation is organized into the following main directories:

```text
RAG-for-generating-image-captioning/
│
├── data/
│   │
│   ├── annotations_trainval2017/
│   │
│   ├── coco_rag_data/
│   │   │
│   │   ├── train2017_subset/
│   │   ├── processed_images/
│   │   ├── processed_images_batch/
│   │   ├── processed_images_test/
│   │   │
│   │   ├── image_embeddings/
│   │   ├── image_embeddings_test/
│   │   │
│   │   ├── train_subset_10000.json
│   │   ├── train_kb_ids.json
│   │   ├── val_ids.json
│   │   ├── test_ids.json
│   │   │
│   │   ├── unified_lookup.json
│   │   ├── test_images_metadata.json
│   │   │
│   │   ├── captions_kb.json
│   │   ├── captions_kb_flat.jsonl
│   │   ├── captions_test.json
│   │   ├── captions_test_flat.jsonl
│   │   │
│   │   ├── text_embeddings_mapping.json
│   │   ├── text_embedding_report.json
│   │   │
│   │   ├── diagnostic_report.json
│   │   ├── image_transformation_report.json
│   │   ├── batch_processing_report.json
│   │   └── caption_cleaning_report.json
│   │
│   └── val2017/
│
├── scripts/
│   ├── download_coco_rag_subset.py
│   ├── diagnose_images.py
│   ├── create_splits.py
│   ├── build_test_metadata.py
│   ├── build_unified_lookup.py
│   ├── image_pipeline.py
│   ├── preprocess_images.py
│   ├── caption_pipeline.py
│   ├── run_preprocessing_batch.py
│   ├── validate_processed_images.py
│   ├── extract_image_embeddings.py
│   ├── extract_text_embeddings.py
│   ├── build_faiss_index.py
│   └── test_phase1_integration.py
│
├── Docs/
│   ├── caption_pipeline.md
│   └── FAISS_Index.md
│
└── notebooks/
    ├── 01-coco-ingestion-audit.ipynb
    └── Report.ipynb
```


## 5. Expected Outcome of Phase 1

At the end of Phase 1, the project should have a structured and reproducible dataset ready for the implementation of the different RAG techniques.

The expected resources include:

- A fixed knowledge base containing 10,000 selected training images.
- Separate validation and test partitions.
- Structured object metadata and caption lookup resources.
- Standardized image files.
- Prepared caption corpora for retrieval and evaluation.
- Image embeddings.
- Text embeddings.
- A FAISS vector index.
- Validation reports confirming the consistency of the generated resources.

These outputs will serve as the common foundation for all RAG techniques evaluated in the project.

Using the same dataset and prepared resources will help ensure that the comparison between techniques is fair and reproducible.

## 6. Conclusion

Phase 1 establishes the data foundation for the comparative study of RAG techniques for image captioning.

It begins with the selection and verification of the MS COCO dataset, followed by the creation of isolated data partitions and the preparation of image metadata and caption resources.

The selected images and captions are then processed to ensure consistency before being transformed into numerical embeddings.

Finally, the vector representations are prepared for indexing and similarity-based retrieval.

By organizing these operations into four steps, the project creates a structured data pipeline that can be reused by all team members during the implementation and evaluation of the different RAG approaches.